# DQN en Acrobot-v1

Implementación de **Deep Q-Network** (Mnih et al., 2015) en PyTorch, basada en
[`cleanrl/dqn.py`](https://docs.cleanrl.dev/rl-algorithms/dqn/), para el TP final de
*Introducción teórico-práctica al Aprendizaje por Refuerzo* (UBA, 2026).

**Cómo usar este notebook**

- Corré las celdas en orden.
- Las celdas con `# TODO` las tenés que completar vos. Cada una dice qué parte del pseudocódigo implementa.
- Después de cada `TODO` hay una celda **✅ Chequeo**: si no tira error, esa parte está bien.
- Primero se prueba todo en **CartPole** (aprende rápido) y después se corre **Acrobot**.

**El algoritmo en una línea:** Q-learning con una red neuronal, más dos parches para que no se rompa:
**replay buffer** (datos casi i.i.d.) y **target network** (target quieto durante C pasos).

## 0. Instalación y configuración

In [ ]:
!pip install -q gymnasium tensorboard

In [ ]:
import random
import time
from dataclasses import dataclass

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter

device = torch.device("cpu")  # para MLPs chicas, la CPU alcanza (y suele ser más rápida)
print("PyTorch", torch.__version__, "| gymnasium", gym.__version__)

**Guardar en Google Drive (recomendado para corridas largas).** Si la sesión de Colab se corta,
lo que no esté en Drive se pierde. Poné `USE_DRIVE = True` para guardar los logs ahí.

In [ ]:
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/tp-dqn-acrobot"
else:
    BASE_DIR = "."

print("Los resultados se guardan en:", BASE_DIR)

## 1. Hiperparámetros

Son los de `cleanrl/dqn.py`. Su relación con el pseudocódigo del paper:

| Parámetro | En el pseudocódigo | Qué es |
|---|---|---|
| `buffer_size` | N | Capacidad del replay buffer D |
| `target_network_frequency` | C | Cada cuántos pasos se copia θ → θ⁻ |
| `batch_size` | — | Tamaño del minibatch que se samplea de D |
| `start_e`, `end_e`, `exploration_fraction` | ε | ε baja linealmente de `start_e` a `end_e` durante esa fracción del entrenamiento |
| `gamma` | γ | Factor de descuento |
| `learning_rate` | α | Paso del optimizador (Adam) |
| `learning_starts` | — | Pasos jugando sin entrenar, para llenar el buffer |
| `train_frequency` | — | Cada cuántos pasos de juego se hace un paso de gradiente |
| `tau` | — | 1.0 = copia completa de la target network (como en el paper) |

In [ ]:
@dataclass
class Args:
    exp_name: str = "dqn"
    env_id: str = "Acrobot-v1"
    seed: int = 1
    total_timesteps: int = 500_000
    learning_rate: float = 2.5e-4
    buffer_size: int = 10_000
    gamma: float = 0.99
    tau: float = 1.0
    target_network_frequency: int = 500
    batch_size: int = 128
    start_e: float = 1.0
    end_e: float = 0.05
    exploration_fraction: float = 0.5
    learning_starts: int = 10_000
    train_frequency: int = 10

Args()

## 2. El entorno

**Acrobot-v1:** dos barras unidas, colgando. Solo se puede aplicar torque en la articulación del medio.
El objetivo es balancear la punta por encima de una línea.

- **Estado** (6 números): cos y sen de los dos ángulos, y las dos velocidades angulares.
  Incluye las velocidades, así que es Markov: no hace falta el preprocesamiento φ del paper.
- **Acciones** (3): torque −1, 0 o +1.
- **Reward:** −1 por cada paso hasta llegar a la meta. Maximizar el retorno = llegar en la menor cantidad de pasos.
- **Fin del episodio:** `terminated` cuando llega a la meta; `truncated` si pasan 500 pasos sin llegar.

In [ ]:
env = gym.make("Acrobot-v1")
obs, info = env.reset(seed=0)
print("Espacio de observaciones:", env.observation_space)
print("Espacio de acciones:     ", env.action_space)
print("Primera observación:     ", obs)

next_obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
print("Reward de un paso:", reward, "| terminated:", terminated, "| truncated:", truncated)
env.close()

## 3. La red Q(s, a; θ)

Una MLP: entra el estado y salen **3 números, un Q por cada acción**.

### TODO 1
Definí `self.network` como una MLP: `obs_dim → 120 → 84 → n_actions`, con ReLU entre capas
(sin activación en la salida: los Q pueden ser negativos).
Pista: `nn.Sequential(nn.Linear(...), nn.ReLU(), ...)`.

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, obs_dim: int, n_actions: int):
        super().__init__()
        # TODO 1: definir la MLP
        self.network = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)

In [ ]:
# ✅ Chequeo TODO 1
_q = QNetwork(obs_dim=6, n_actions=3)
_out = _q(torch.zeros(5, 6))
assert _out.shape == (5, 3), f"La salida debería ser (5, 3) y es {tuple(_out.shape)}"
print("TODO 1 OK. Parámetros de la red:", sum(p.numel() for p in _q.parameters()))

## 4. El replay buffer D

Guarda transiciones `(s, a, r, s', terminated)`. Cuando se llena, las nuevas pisan a las más viejas.
`add` ya está hecho.

### TODO 2
Completá `sample`: elegí `batch_size` índices **al azar** entre las transiciones guardadas
(entre 0 y `self.size`). Samplear al azar es lo que rompe la correlación entre pasos consecutivos.
Pista: `np.random.randint`.

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity: int, obs_dim: int):
        self.capacity = capacity
        self.obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.next_obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.terminated = np.zeros(capacity, dtype=np.float32)
        self.pos = 0   # dónde se escribe la próxima transición
        self.size = 0  # cuántas transiciones válidas hay

    def add(self, obs, action, reward, next_obs, terminated):
        self.obs[self.pos] = obs
        self.actions[self.pos] = action
        self.rewards[self.pos] = reward
        self.next_obs[self.pos] = next_obs
        self.terminated[self.pos] = float(terminated)
        self.pos = (self.pos + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size: int) -> dict:
        # TODO 2: índices al azar entre 0 y self.size
        idx = None
        return {
            "obs": torch.as_tensor(self.obs[idx], device=device),
            "actions": torch.as_tensor(self.actions[idx], device=device),
            "rewards": torch.as_tensor(self.rewards[idx], device=device),
            "next_obs": torch.as_tensor(self.next_obs[idx], device=device),
            "terminated": torch.as_tensor(self.terminated[idx], device=device),
        }

In [ ]:
# ✅ Chequeo TODO 2
_rb = ReplayBuffer(capacity=50, obs_dim=6)
for i in range(1, 31):  # 30 transiciones, con valores 1..30 (las posiciones vacías valen 0)
    _rb.add(np.full(6, i), i % 3, -1.0, np.full(6, i + 1), False)
_b = _rb.sample(64)
assert _b["obs"].shape == (64, 6) and _b["actions"].shape == (64,)
assert _b["obs"][:, 0].min() > 0, "Estás sampleando posiciones vacías del buffer (usá self.size, no self.capacity)"
assert len(set(_b["obs"][:, 0].tolist())) > 1, "Los índices no parecen aleatorios"
print("TODO 2 OK")

## 5. Exploración: ε-greedy

`linear_schedule` ya está hecho: devuelve el ε de cada paso (baja linealmente y después queda fijo en `end_e`).

### TODO 3
Completá `select_action`:
- con probabilidad `epsilon`, una acción **al azar** (`env.action_space.sample()`);
- si no, la acción con **mayor Q** según la red: `argmax_a Q(s, a; θ)`.

Pista: la red espera un batch, así que pasale `obs` como tensor de forma `(1, obs_dim)`.
Para elegir la acción no hace falta gradiente (`torch.no_grad()`).

In [ ]:
def linear_schedule(start_e: float, end_e: float, duration: int, t: int) -> float:
    slope = (end_e - start_e) / duration
    return max(slope * t + start_e, end_e)


def select_action(q_network: QNetwork, obs: np.ndarray, epsilon: float, env) -> int:
    # TODO 3: epsilon-greedy
    action = None
    return int(action)

In [ ]:
# ✅ Chequeo TODO 3
_env = gym.make("Acrobot-v1")
_obs, _ = _env.reset(seed=0)
_q = QNetwork(6, 3)
_greedy = int(_q(torch.as_tensor(_obs).unsqueeze(0)).argmax())
assert all(select_action(_q, _obs, 0.0, _env) == _greedy for _ in range(20)), "Con epsilon=0 tiene que elegir siempre el argmax"
assert len({select_action(_q, _obs, 1.0, _env) for _ in range(200)}) == 3, "Con epsilon=1 tiene que elegir al azar entre las 3 acciones"
print("TODO 3 OK. Epsilon a mitad de la exploración:", linear_schedule(1.0, 0.05, 1000, 500))

## 6. El target y la loss

Es el corazón del algoritmo: la regla de *Approximate Q-learning*.

$$y_j = r_j + \gamma\,(1 - \text{terminated}_j)\,\max_{a'} \hat{Q}(s'_j, a'; \theta^-)$$

$$L(\theta) = \frac{1}{B}\sum_j \big(y_j - Q(s_j, a_j; \theta)\big)^2$$

### TODO 4: `compute_targets`
- Usá la **target network** (θ⁻), no la principal.
- Si `terminated = 1`, el estado siguiente es terminal: no hay futuro y el target es solo el reward.
- Calculalo **sin gradiente** (`torch.no_grad()`): es el semi-gradiente, el target se trata como constante.

### TODO 5: `compute_loss`
- La red devuelve Q para **las 3 acciones**; quedate solo con el Q de la acción que se tomó en cada transición.
  Pista: `q_values.gather(1, actions.unsqueeze(1)).squeeze(1)`.
- Loss: MSE entre ese Q y los targets (`F.mse_loss`).
- Devolvé también `q_sa`, para poder graficar cómo evolucionan los Q.

In [ ]:
def compute_targets(target_network: QNetwork, rewards, next_obs, terminated, gamma: float) -> torch.Tensor:
    # TODO 4
    targets = None
    return targets


def compute_loss(q_network: QNetwork, obs, actions, targets):
    # TODO 5
    q_sa = None
    loss = None
    return loss, q_sa

In [ ]:
# ✅ Chequeo TODO 4 y 5
class _FixedQ(nn.Module):
    # Red de mentira: devuelve siempre Q = [1, 5, 2] para cualquier estado
    def forward(self, x):
        return torch.tensor([[1.0, 5.0, 2.0]]).repeat(x.shape[0], 1)

_r = torch.tensor([-1.0, -1.0])
_next = torch.zeros(2, 6)
_term = torch.tensor([0.0, 1.0])
_y = compute_targets(_FixedQ(), _r, _next, _term, gamma=0.9)
assert torch.allclose(_y, torch.tensor([-1.0 + 0.9 * 5.0, -1.0])), f"Targets esperados [3.5, -1.0], obtuve {_y}"

_loss, _qsa = compute_loss(_FixedQ(), torch.zeros(2, 6), torch.tensor([1, 2]), torch.tensor([5.0, 0.0]))
assert torch.allclose(_qsa, torch.tensor([5.0, 2.0])), f"Q(s,a) esperados [5, 2], obtuve {_qsa}"
assert torch.isclose(_loss, torch.tensor(2.0)), f"Loss esperada 2.0, obtuve {_loss}"
print("TODO 4 y 5 OK")

## 7. El loop de entrenamiento

Sigue el pseudocódigo del paper. Cada paso del juego:

1. ε-greedy elige una acción y se ejecuta en el entorno.
2. Se guarda la transición en D.
3. Cada `train_frequency` pasos (y una vez superado `learning_starts`): minibatch de D → targets → loss → paso de gradiente.
4. Cada C pasos: se actualiza la target network.

### TODO 6: inicializar la target network como copia de la principal (θ⁻ = θ)
Pista: `load_state_dict(... .state_dict())`.

### TODO 7: guardar la transición en el buffer
Ojo: guardá `terminated`, **no** `truncated`. Si el episodio se cortó por tiempo (500 pasos),
el estado no es terminal y su futuro tiene que contar en el target.

### TODO 8: paso de gradiente
Los tres pasos de siempre en PyTorch: poner gradientes en cero, backprop, paso del optimizador.

### TODO 9: actualizar la target network
Con `tau = 1` es copiar θ → θ⁻. En general (soft update): θ⁻ ← τ·θ + (1 − τ)·θ⁻, parámetro por parámetro.
Pista: recorré `zip(target_network.parameters(), q_network.parameters())` y usá `.data.copy_(...)`.

In [ ]:
def train(args: Args, verbose: bool = True):
    run_name = f"{args.env_id}__{args.exp_name}__seed{args.seed}__{int(time.time())}"
    writer = SummaryWriter(f"{BASE_DIR}/runs/{run_name}")

    # Semillas, para que cada corrida sea reproducible
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)

    env = gym.make(args.env_id)
    env.action_space.seed(args.seed)
    obs_dim = env.observation_space.shape[0]
    n_actions = env.action_space.n

    q_network = QNetwork(obs_dim, n_actions).to(device)
    optimizer = optim.Adam(q_network.parameters(), lr=args.learning_rate)
    target_network = QNetwork(obs_dim, n_actions).to(device)
    # TODO 6: θ⁻ = θ

    rb = ReplayBuffer(args.buffer_size, obs_dim)
    logs = {"episode_steps": [], "episode_returns": [], "train_steps": [], "losses": [], "q_values": []}

    obs, _ = env.reset(seed=args.seed)
    ep_return, ep_len = 0.0, 0
    exploration_steps = int(args.exploration_fraction * args.total_timesteps)
    start_time = time.time()

    for global_step in range(args.total_timesteps):
        # --- 1. Actuar ---
        epsilon = linear_schedule(args.start_e, args.end_e, exploration_steps, global_step)
        action = select_action(q_network, obs, epsilon, env)
        next_obs, reward, terminated, truncated, _ = env.step(action)
        ep_return += reward
        ep_len += 1

        # --- 2. Guardar en D ---
        # TODO 7

        obs = next_obs
        if terminated or truncated:
            logs["episode_steps"].append(global_step)
            logs["episode_returns"].append(ep_return)
            writer.add_scalar("charts/episodic_return", ep_return, global_step)
            writer.add_scalar("charts/episodic_length", ep_len, global_step)
            writer.add_scalar("charts/epsilon", epsilon, global_step)
            if verbose and len(logs["episode_returns"]) % 50 == 0:
                recent = np.mean(logs["episode_returns"][-50:])
                print(f"paso {global_step:>7} | episodios {len(logs['episode_returns']):>5} | "
                      f"retorno medio (últimos 50) {recent:8.1f} | epsilon {epsilon:.2f}")
            obs, _ = env.reset()
            ep_return, ep_len = 0.0, 0

        # --- 3. Aprender ---
        if global_step > args.learning_starts and global_step % args.train_frequency == 0:
            batch = rb.sample(args.batch_size)
            targets = compute_targets(target_network, batch["rewards"], batch["next_obs"],
                                      batch["terminated"], args.gamma)
            loss, q_sa = compute_loss(q_network, batch["obs"], batch["actions"], targets)

            # TODO 8: paso de gradiente

            if global_step % 100 == 0:
                logs["train_steps"].append(global_step)
                logs["losses"].append(loss.item())
                logs["q_values"].append(q_sa.mean().item())
                writer.add_scalar("losses/td_loss", loss.item(), global_step)
                writer.add_scalar("losses/q_values", q_sa.mean().item(), global_step)

        # --- 4. Actualizar la target network ---
        if global_step % args.target_network_frequency == 0:
            # TODO 9
            pass

    env.close()
    writer.close()
    if verbose:
        print(f"Listo en {(time.time() - start_time) / 60:.1f} minutos")
    return q_network, logs

In [ ]:
def moving_average(x, window=50):
    x = np.asarray(x, dtype=float)
    if len(x) < window:
        return x
    return np.convolve(x, np.ones(window) / window, mode="valid")


def plot_run(logs, title=""):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    steps = np.asarray(logs["episode_steps"])
    returns = logs["episode_returns"]
    axes[0].plot(steps, returns, alpha=0.25, label="por episodio")
    ma = moving_average(returns)
    axes[0].plot(steps[len(steps) - len(ma):], ma, label="media móvil (50)")
    axes[0].set(title="Retorno por episodio", xlabel="paso", ylabel="retorno")
    axes[0].legend()
    axes[1].plot(logs["train_steps"], logs["losses"])
    axes[1].set(title="Loss TD", xlabel="paso", yscale="log")
    axes[2].plot(logs["train_steps"], logs["q_values"])
    axes[2].set(title="Q(s, a) medio del minibatch", xlabel="paso")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

## 8. Prueba rápida: CartPole

Antes de Acrobot, chequeamos que todo funcione en CartPole, que aprende rápido.
CartPole da +1 por cada paso que el palo se mantiene en pie (máximo 500).
Si todo está bien, el retorno medio debería subir claramente durante el entrenamiento.
Si queda plano cerca de 10–20, algo de los TODO está mal.

In [ ]:
args_cp = Args(env_id="CartPole-v1", total_timesteps=100_000)
_, logs_cp = train(args_cp)
plot_run(logs_cp, "CartPole-v1, seed 1")

## 9. Acrobot-v1

Ahora sí, el entorno del TP. Arranca en −500 (nunca llega a la meta en 500 pasos) y,
si aprende, el retorno sube hacia valores mucho menos negativos (llega en pocos pasos).
Para comparar: el modelo de CleanRL `Acrobot-v1-dqn-seed1` que linkea la consigna.

Con 500.000 pasos tarda un rato. Para una primera prueba podés bajar `total_timesteps`.

In [ ]:
args_ac = Args(env_id="Acrobot-v1", seed=1)
q_ac, logs_ac = train(args_ac)
plot_run(logs_ac, "Acrobot-v1, seed 1")

### Curvas en TensorBoard
Todo lo que se loguea durante el entrenamiento se puede ver acá, actualizado en vivo.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {BASE_DIR}/runs

## 10. Varias seeds

Una sola corrida no alcanza: el resultado depende mucho de la seed.
Acá se corren varias y se grafica la media ± el desvío estándar.

In [ ]:
def returns_on_grid(logs, grid):
    # Interpola el retorno (media móvil) sobre una grilla común de pasos, para poder promediar seeds
    steps = np.asarray(logs["episode_steps"])
    ma = moving_average(logs["episode_returns"])
    return np.interp(grid, steps[len(steps) - len(ma):], ma)


SEEDS = [1, 2, 3]
all_logs = {}
for seed in SEEDS:
    print(f"\n=== seed {seed} ===")
    _, all_logs[seed] = train(Args(env_id="Acrobot-v1", seed=seed), verbose=False)

grid = np.linspace(Args().learning_starts, Args().total_timesteps, 200)
curves = np.array([returns_on_grid(all_logs[s], grid) for s in SEEDS])
mean, std = curves.mean(axis=0), curves.std(axis=0)

plt.figure(figsize=(8, 4))
plt.plot(grid, mean, label=f"media de {len(SEEDS)} seeds")
plt.fill_between(grid, mean - std, mean + std, alpha=0.3, label="± 1 desvío")
plt.xlabel("paso"); plt.ylabel("retorno (media móvil 50)"); plt.title("DQN en Acrobot-v1")
plt.legend(); plt.show()

## 11. Ideas para ir más allá

Cuando lo de arriba funcione:

- **Ablaciones:** correr sin target network (`target_network_frequency=1`), con buffer chico
  (`buffer_size=batch_size`), sin `learning_starts`. ¿Qué componente importa más?
- **Double DQN:** cambiar una línea de `compute_targets` (la red online elige la acción, la target la evalúa)
  y comparar los Q-values estimados.
- **Analizar los Q-values** que aprende la red en distintos estados.
- **El pasito más:** `highway-fast-v0` (paquete `highway-env`).